# Proyecto 6. Analisis de resultados

FTL de comparacion por plano anatomico: **FedAvg vs FedProx vs FedAvgM**.

Tres familias del Tema 8 de los apuntes (T8.4.11): base (FedAvg, T7.4), correccion cliente (FedProx, T8.2), correccion servidor (FedAvgM, T8.3.3).

Metricas alineadas con T9.4.7 (Accuracy, F1, AUC-ROC, Sensibilidad, Especificidad), T9.4.6 (equidad: sigma, peor cliente, brecha) y T9.4.8 (brechas Delta_fed-local y Delta_cent-fed).

El cuaderno se lee de arriba abajo siguiendo la estructura de un articulo: primero datos, luego configuracion, despues resultados, finalmente coste y limitaciones.

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.metrics import (
    average_precision_score, brier_score_loss,
    f1_score, roc_auc_score, recall_score, accuracy_score,
)

RESULTS = Path('../results')
CSVS = Path('../../federated_csvs/06_hfl_cls_plane_comparison_flair')
sns.set_theme(style='whitegrid', palette='tab10')
HOSPITALS = ['H1-axial', 'H2-coronal', 'H3-sagital', 'H4-mixto']

In [ ]:
def load(name):
    with open(RESULTS / f'{name}.json') as f:
        return json.load(f)

def load_optional(name):
    p = RESULTS / f'{name}.json'
    return json.load(open(p)) if p.exists() else None

local        = load('local_only')
central      = load('centralized')
fedavg       = load('fedavg')
fedprox      = load('fedprox')
fedavgm      = load_optional('fedavgm')

ALG_LIST = [('FedAvg', fedavg), ('FedProx', fedprox)]
if fedavgm is not None:
    ALG_LIST.append(('FedAvgM', fedavgm))

# Bloque 1. Datos y contexto del escenario federado

## 1. Datos y particiones

Antes de mirar resultados, conviene caracterizar el dataset que cada cliente ve. La particion ya viene fijada en `federated_csvs/06_hfl_cls_plane_comparison_flair/`.

In [ ]:
# Resumen por hospital y test global
hospital_summary = pd.read_csv(CSVS / 'hospital_summary.csv')
display(hospital_summary)

train_dfs = [pd.read_csv(CSVS / f'hospital_{k}_train.csv') for k in range(1, 5)]
test_df   = pd.read_csv(CSVS / 'global_test.csv')

filas_data = []
for k, df in enumerate(train_dfs, 1):
    filas_data.append({
        'Hospital': HOSPITALS[k-1],
        'Plano': df['plane_profile'].iloc[0],
        'Pacientes': df['patient_id'].nunique(),
        'Slices': len(df),
        'Ratio EM (slice)': f"{(df['label_id']==1).mean():.3f}",
        'EM pacientes': df[df.label_id==1]['patient_id'].nunique(),
        'Control pacientes': df[df.label_id==0]['patient_id'].nunique(),
    })
filas_data.append({
    'Hospital': 'Global test',
    'Plano': 'mixto',
    'Pacientes': test_df['patient_id'].nunique(),
    'Slices': len(test_df),
    'Ratio EM (slice)': f"{(test_df['label_id']==1).mean():.3f}",
    'EM pacientes': test_df[test_df.label_id==1]['patient_id'].nunique(),
    'Control pacientes': test_df[test_df.label_id==0]['patient_id'].nunique(),
})
pd.DataFrame(filas_data)

In [ ]:
# Verificacion: pacientes disjuntos entre hospitales y entre train/test
pid_train = {k: set(df['patient_id'].unique()) for k, df in enumerate(train_dfs, 1)}
pid_test  = set(test_df['patient_id'].unique())

print('Overlap entre hospitales (intra-train):')
for i in range(1, 5):
    for j in range(i+1, 5):
        print(f'  H{i} & H{j}: {len(pid_train[i] & pid_train[j])} pacientes en comun')

print()
print('Overlap train/test:')
for k in range(1, 5):
    print(f'  H{k} train & global test: {len(pid_train[k] & pid_test)}')

print()
print('Sin overlap = particion limpia, sin contaminacion entre clientes ni entre train/test.')

## 2. Heterogeneidad medida entre planos

Aviso sobre el preprocesado del dataset: las imagenes ya vienen normalizadas con **Kernel Density Estimation (KDE)** por el equipo que publica el conjunto de datos. Las carpetas se llaman `MRIms_kde/` y `MRIcontrol_kde/` (en `p6/data/`), y el PDF *DescripcionConjuntoDatos_2025_26* lo confirma: "las imagenes han sido previamente normalizadas y preprocesadas para garantizar una escala de intensidades coherente". **No es decision nuestra**; es una condicion de partida.

El KDE iguala las distribuciones marginales de intensidad entre imagenes (su objetivo: que un mismo tejido tenga niveles de gris similares en todas las adquisiciones). Eso explica que las distancias sobre histogramas marginales sean casi cero, aunque las imagenes sean visualmente muy distintas entre planos.

Para no quedarse en esa vista enganosa, complementamos con distancias en el espacio de **embeddings de ResNet-18** (T6.6 considera metricas en gradientes y representaciones). El espacio de embeddings si captura la estructura espacial que el KDE no toca, y refleja la heterogeneidad real entre clientes.

1. **Histograma de intensidad de pixel** (TV, Hellinger, KL): heterogeneidad marginal de gris.
2. **Distancia entre embeddings de ResNet-18** (||mu_a-mu_b||, Frechet, cosine): heterogeneidad estructural.

In [ ]:
het_path = RESULTS / 'heterogeneity.json'
if not het_path.exists():
    print('Falta results/heterogeneity.json. Ejecuta:  python scripts/diagnose_heterogeneity.py')
else:
    het = json.load(open(het_path))

    pares_planos = pd.DataFrame([
        {'Par': k, 'TV': v['TV'], 'Hellinger': v['Hellinger'], 'KL': v['KL']}
        for k, v in het['plane_distances'].items()
    ])
    pares_hosp = pd.DataFrame([
        {'Par': k, 'TV': v['TV'], 'Hellinger': v['Hellinger'], 'KL': v['KL']}
        for k, v in het['hospital_distances'].items()
    ])
    print('Distancias entre planos (histograma de intensidad):')
    display(pares_planos.style.format({'TV': '{:.4f}', 'Hellinger': '{:.4f}', 'KL': '{:.4f}'}))
    print()
    print('Distancias entre hospitales (histograma de intensidad):')
    display(pares_hosp.style.format({'TV': '{:.4f}', 'Hellinger': '{:.4f}', 'KL': '{:.4f}'}))

    if 'plane_distances_embeddings' in het:
        emb_planos = pd.DataFrame([
            {'Par': k, '||mu_a-mu_b||': v['mean_dist'], 'Frechet': v['frechet'], 'cosine(mu_a,mu_b)': v['cosine']}
            for k, v in het['plane_distances_embeddings'].items()
        ])
        emb_hosp = pd.DataFrame([
            {'Par': k, '||mu_a-mu_b||': v['mean_dist'], 'Frechet': v['frechet'], 'cosine(mu_a,mu_b)': v['cosine']}
            for k, v in het['hospital_distances_embeddings'].items()
        ])
        print()
        print('Distancias entre planos (embeddings ResNet-18):')
        display(emb_planos.style.format({'||mu_a-mu_b||': '{:.3f}', 'Frechet': '{:.3f}', 'cosine(mu_a,mu_b)': '{:.4f}'}))
        print()
        print('Distancias entre hospitales (embeddings ResNet-18):')
        display(emb_hosp.style.format({'||mu_a-mu_b||': '{:.3f}', 'Frechet': '{:.3f}', 'cosine(mu_a,mu_b)': '{:.4f}'}))
    else:
        print()
        print('No hay distancias en espacio de embeddings en el JSON.')
        print('Vuelve a ejecutar  python scripts/diagnose_heterogeneity.py  para anadirlas.')

    img_path = RESULTS / 'heterogeneity_hist.png'
    if img_path.exists():
        from IPython.display import Image
        display(Image(filename=str(img_path)))

## 3. Visualizacion del dataset

Muestras visuales generadas por `scripts/visualize_dataset.py`. Si los PNGs no estan, ejecuta el script primero:

```
python scripts/visualize_dataset.py
```

In [ ]:
from IPython.display import Image, display
for fname, titulo in [
    ('comparacion_planos.png',     'Comparacion de los tres planos anatomicos'),
    ('muestras_por_hospital.png',  'Muestras del dataset por hospital y clase'),
    ('distribucion_clases.png',    'Distribucion de clases por hospital'),
]:
    p = RESULTS / fname
    if p.exists():
        print(titulo)
        display(Image(filename=str(p)))
    else:
        print(f'(falta {fname} - ejecuta scripts/visualize_dataset.py)')

# Bloque 2. Configuracion experimental

## 4. Algoritmos, baselines y presupuesto

**Algoritmos federados** (tres familias del Tema 8):

| Familia | Algoritmo | Donde corrige | Hiperparam |
|---|---|---|---|
| Base | FedAvg (T7.4) | --- | --- |
| Correccion cliente | FedProx (T8.2) | Termino proximal mu/2 ||w-w_global||^2 | mu = 0.1 |
| Correccion servidor | FedAvgM (T8.3.3) | Momento beta sobre el pseudo-gradiente | beta = 0.9, eta_s = 1.0 |

**Baselines** (T9.3.7):

| Baseline | Descripcion | Rol |
|---|---|---|
| Local-only | Cada hospital entrena solo, sin colaborar | Cota inferior |
| Centralizado | Todos los datos en un servidor | Cota superior |

**Presupuesto comun a todos los experimentos federados**:

```
R = 15 rondas      E = 5 epocas locales por ronda
B = 32 batch       eta = 1e-4 (Adam)
C = 1.0 (todos los clientes en cada ronda)
seeds = [42, 123]  -> media y desviacion sobre 2 seeds
```

**Modelo**: ResNet-18 preentrenada en ImageNet, con la capa final reemplazada por `Linear(512, 1)` y `BCEWithLogitsLoss`. Las imagenes en escala de grises se replican a 3 canales para compatibilidad con la red preentrenada.

**Metricas reportadas**: AUC-ROC, F1, Sensibilidad, Especificidad, Accuracy (T9.4.7) — globales y por hospital. Mas σ(AUC) entre clientes, peor cliente y brechas Delta_fed-local, Delta_cent-fed (T9.4.6 y T9.4.8). Mas AUPRC, Brier y ECE post-hoc desde las probabilidades crudas.

# Bloque 3. Resultados principales

## 5. Tabla resumen baselines vs federado

In [ ]:
rows = []

for c in local['clients']:
    rows.append({
        'Metodo': 'Local-only', 'Hospital': HOSPITALS[c['hospital']-1],
        'AUC': f"{c['auc']:.4f}", 'F1': f"{c['f1']:.4f}",
        'Sen': f"{c['sensitivity']:.4f}",
        'Spe': f"{c.get('specificity', float('nan')):.4f}",
        'Acc': f"{c.get('accuracy', float('nan')):.4f}",
    })

cg = central['final_global']
rows.append({
    'Metodo': 'Centralizado', 'Hospital': 'Global',
    'AUC': f"{cg['auc']:.4f}", 'F1': f"{cg['f1']:.4f}",
    'Sen': f"{cg['sensitivity']:.4f}",
    'Spe': f"{cg.get('specificity', float('nan')):.4f}",
    'Acc': f"{cg.get('accuracy', float('nan')):.4f}",
})

for nombre, exp in ALG_LIST:
    agg = exp['aggregated']
    def gs(key):
        m = agg.get(f'global_{key}', None)
        return f"{m['mean']:.4f} ± {m['std']:.4f}" if m else '-'
    rows.append({
        'Metodo': nombre, 'Hospital': 'Global',
        'AUC': gs('auc'), 'F1': gs('f1'),
        'Sen': gs('sensitivity'),
        'Spe': gs('specificity'),
        'Acc': gs('accuracy'),
    })

df_resumen = pd.DataFrame(rows)
df_resumen.to_csv(RESULTS / 'tabla_resumen.csv', index=False)
df_resumen

## 6. Tabla maestra Hospital x Algoritmo x Metrica

In [ ]:
def fmt_mean_std(values):
    arr = np.array(values, dtype=float)
    arr = arr[~np.isnan(arr)]
    if len(arr) == 0:
        return '-'
    if len(arr) == 1:
        return f"{arr[0]:.4f}"
    return f"{arr.mean():.4f} ± {arr.std():.4f}"

filas = []

for c in local['clients']:
    filas.append({
        'Algoritmo': 'Local-only', 'Hospital': HOSPITALS[c['hospital']-1],
        'AUC': f"{c['auc']:.4f}", 'F1': f"{c['f1']:.4f}",
        'Sen': f"{c['sensitivity']:.4f}",
        'Spe': f"{c.get('specificity', float('nan')):.4f}",
        'Acc': f"{c.get('accuracy', float('nan')):.4f}",
    })

for k, m in enumerate(central['final_clients']):
    filas.append({
        'Algoritmo': 'Centralizado', 'Hospital': HOSPITALS[k],
        'AUC': f"{m['auc']:.4f}", 'F1': f"{m['f1']:.4f}",
        'Sen': f"{m['sensitivity']:.4f}",
        'Spe': f"{m.get('specificity', float('nan')):.4f}",
        'Acc': f"{m.get('accuracy', float('nan')):.4f}",
    })
filas.append({
    'Algoritmo': 'Centralizado', 'Hospital': 'Global',
    'AUC': f"{central['final_global']['auc']:.4f}",
    'F1':  f"{central['final_global']['f1']:.4f}",
    'Sen': f"{central['final_global']['sensitivity']:.4f}",
    'Spe': f"{central['final_global'].get('specificity', float('nan')):.4f}",
    'Acc': f"{central['final_global'].get('accuracy', float('nan')):.4f}",
})

for nombre, exp in ALG_LIST:
    for k in range(4):
        filas.append({
            'Algoritmo': nombre, 'Hospital': HOSPITALS[k],
            'AUC': fmt_mean_std([s['final_clients'][k]['auc'] for s in exp['seeds']]),
            'F1':  fmt_mean_std([s['final_clients'][k]['f1']  for s in exp['seeds']]),
            'Sen': fmt_mean_std([s['final_clients'][k]['sensitivity'] for s in exp['seeds']]),
            'Spe': fmt_mean_std([s['final_clients'][k].get('specificity', float('nan')) for s in exp['seeds']]),
            'Acc': fmt_mean_std([s['final_clients'][k].get('accuracy',    float('nan')) for s in exp['seeds']]),
        })
    filas.append({
        'Algoritmo': nombre, 'Hospital': 'Global',
        'AUC': fmt_mean_std([s['final_global']['auc'] for s in exp['seeds']]),
        'F1':  fmt_mean_std([s['final_global']['f1']  for s in exp['seeds']]),
        'Sen': fmt_mean_std([s['final_global']['sensitivity'] for s in exp['seeds']]),
        'Spe': fmt_mean_std([s['final_global'].get('specificity', float('nan')) for s in exp['seeds']]),
        'Acc': fmt_mean_std([s['final_global'].get('accuracy',    float('nan')) for s in exp['seeds']]),
    })

tabla_master = pd.DataFrame(filas)
tabla_master.to_csv(RESULTS / 'tabla_maestra.csv', index=False)
print(f"Guardada en {RESULTS / 'tabla_maestra.csv'}")
tabla_master

# Bloque 4. Analisis por hospital y equidad

## 7. F1 y Sensibilidad por hospital

AUC global satura cerca de 1 en esta tarea. La diferencia entre algoritmos se ve mejor en F1 y sensibilidad por cliente, especialmente en H1, H2, H3 (un solo plano) frente a H4 (mixto).

El train tiene ratio EM ~0.605 y el test global ~0.459, asi que el F1 a threshold 0.5 esta penalizado por descalibracion (la magnitud absoluta hay que leerla con cuidado; las **diferencias entre algoritmos** si son comparables).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
x = np.arange(4)
n_alg = len(ALG_LIST)
width = 0.8 / n_alg

for col, metric, ylabel in [(0, 'f1', 'F1-score'), (1, 'sensitivity', 'Sensibilidad (recall MS)')]:
    ax = axes[col]
    for i, (nombre, exp) in enumerate(ALG_LIST):
        client_vals = np.mean(
            [[s['final_clients'][k][metric] for k in range(4)] for s in exp['seeds']], axis=0
        )
        client_std = np.std(
            [[s['final_clients'][k][metric] for k in range(4)] for s in exp['seeds']], axis=0
        )
        ax.bar(x + i*width - 0.4 + width/2, client_vals, width, yerr=client_std, capsize=4, label=nombre)
    ax.set_xticks(x); ax.set_xticklabels(HOSPITALS)
    ax.set_ylabel(ylabel)
    ax.set_title(f'{ylabel} por hospital - media ± std seeds')
    ax.legend(); ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(RESULTS / 'f1_sen_por_hospital.png', dpi=150)
plt.show()

## 8. AUC por hospital en la ultima ronda

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
x = np.arange(4)
n_alg = len(ALG_LIST)
width = 0.8 / n_alg

for i, (nombre, exp) in enumerate(ALG_LIST):
    client_aucs = np.mean(
        [[s['final_clients'][k]['auc'] for k in range(4)] for s in exp['seeds']], axis=0
    )
    client_std = np.std(
        [[s['final_clients'][k]['auc'] for k in range(4)] for s in exp['seeds']], axis=0
    )
    ax.bar(x + i*width - 0.4 + width/2, client_aucs, width, yerr=client_std, capsize=4, label=nombre)

ax.set_xticks(x); ax.set_xticklabels(HOSPITALS)
ax.set_ylabel('AUC-ROC')
ax.set_title('AUC por hospital (ultima ronda, media ± std seeds)')
ax.legend(); ax.set_ylim(0.5, 1.02)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(RESULTS / 'auc_por_hospital.png', dpi=150)
plt.show()

## 9. Equidad entre hospitales: sigma(AUC) por ronda

Tres vistas: auto-escala, lineal con zoom y logaritmica. La auto-escala suele aplastar la curva cuando los valores estan cerca de 0; las otras dos vistas hacen visible la diferencia fina entre algoritmos.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))

def mean_std(exp, key):
    vals = np.array([s['history'][key] for s in exp['seeds']])
    return vals.mean(axis=0), vals.std(axis=0)

rounds_full = fedavg['seeds'][0]['history']['round']

all_eq = []
for _, exp in ALG_LIST:
    all_eq.extend([v for s in exp['seeds'] for v in s['history']['equity_auc']])
y_top = max(np.percentile(all_eq, 99) * 1.2, 1e-3)

linestyles = ['-', '--', ':']
for (nombre, exp), ls in zip(ALG_LIST, linestyles):
    m, s = mean_std(exp, 'equity_auc')
    for ax in axes:
        ax.plot(rounds_full, m, ls=ls, lw=2, label=nombre)
        ax.fill_between(rounds_full, np.maximum(m - s, 1e-6), m + s, alpha=0.2)

axes[0].set_title('sigma(AUC) - auto-escala'); axes[0].set_xlabel('Ronda'); axes[0].set_ylabel('sigma(AUC)')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].set_title('sigma(AUC) - lineal con zoom')
axes[1].set_ylim(0, y_top)
axes[1].set_xlabel('Ronda'); axes[1].set_ylabel('sigma(AUC)')
axes[1].legend(); axes[1].grid(alpha=0.3)

axes[2].set_title('sigma(AUC) - escala log')
axes[2].set_yscale('log')
axes[2].set_xlabel('Ronda'); axes[2].set_ylabel('sigma(AUC) (log)')
axes[2].legend(); axes[2].grid(alpha=0.3, which='both')

plt.tight_layout()
plt.savefig(RESULTS / 'equity.png', dpi=150)
plt.show()

## 10. Equidad operativa: peor cliente, ratio worst/best, brecha mediana-peor

T9.4.6 pide reportar mas de un resumen de equidad. La media oculta a quien queda mal servido.

In [ ]:
filas_eq = []
for nombre, exp in ALG_LIST:
    aucs_per_seed_per_client = np.array(
        [[s['final_clients'][k]['auc'] for k in range(4)] for s in exp['seeds']]
    )
    aucs_mean_per_client = aucs_per_seed_per_client.mean(axis=0)
    f1_per_seed_per_client = np.array(
        [[s['final_clients'][k]['f1'] for k in range(4)] for s in exp['seeds']]
    )
    f1_mean_per_client = f1_per_seed_per_client.mean(axis=0)

    filas_eq.append({
        'Algoritmo': nombre,
        'AUC media': f"{aucs_mean_per_client.mean():.4f}",
        'AUC peor':  f"{aucs_mean_per_client.min():.4f}",
        'AUC mejor': f"{aucs_mean_per_client.max():.4f}",
        'AUC ratio peor/mejor': f"{aucs_mean_per_client.min() / aucs_mean_per_client.max():.4f}",
        'AUC brecha mediana-peor': f"{np.median(aucs_mean_per_client) - aucs_mean_per_client.min():+.4f}",
        'F1 peor':   f"{f1_mean_per_client.min():.4f}",
        'F1 brecha mediana-peor': f"{np.median(f1_mean_per_client) - f1_mean_per_client.min():+.4f}",
    })

tabla_eq = pd.DataFrame(filas_eq)
tabla_eq.to_csv(RESULTS / 'equidad_operativa.csv', index=False)
display(tabla_eq)

## 11. Brechas como narrativa: Delta_fed-local y Delta_cent-fed

T9.4.8 dice que lo importante para el informe **no es el numero absoluto, sino las brechas**:

- Delta_fed-local: cuanto mejora cada cliente al federar respecto a entrenar solo. Si es ~0, la federacion no aporta valor.
- Delta_cent-fed: cuanto se pierde por no centralizar. Pequeno = federacion competitiva.

In [ ]:
local_auc = {c['hospital']: c['auc'] for c in local['clients']}
local_f1  = {c['hospital']: c['f1']  for c in local['clients']}
cent_auc = {k+1: m['auc'] for k, m in enumerate(central['final_clients'])}
cent_f1  = {k+1: m['f1']  for k, m in enumerate(central['final_clients'])}

filas_brecha = []
for nombre, exp in ALG_LIST:
    fed_auc_per_h = np.mean(
        [[s['final_clients'][k]['auc'] for k in range(4)] for s in exp['seeds']], axis=0
    )
    fed_f1_per_h = np.mean(
        [[s['final_clients'][k]['f1'] for k in range(4)] for s in exp['seeds']], axis=0
    )
    for k in range(4):
        h = k + 1
        filas_brecha.append({
            'Algoritmo': nombre, 'Hospital': HOSPITALS[k],
            'AUC fed': f"{fed_auc_per_h[k]:.4f}",
            'd_fed-local AUC': f"{fed_auc_per_h[k] - local_auc[h]:+.4f}",
            'd_cent-fed AUC':  f"{cent_auc[h] - fed_auc_per_h[k]:+.4f}",
            'F1 fed': f"{fed_f1_per_h[k]:.4f}",
            'd_fed-local F1': f"{fed_f1_per_h[k] - local_f1[h]:+.4f}",
            'd_cent-fed F1':  f"{cent_f1[h] - fed_f1_per_h[k]:+.4f}",
        })

tabla_brecha = pd.DataFrame(filas_brecha)
tabla_brecha.to_csv(RESULTS / 'brechas.csv', index=False)
display(tabla_brecha)

fig, axes = plt.subplots(2, len(ALG_LIST), figsize=(5 * len(ALG_LIST), 8), squeeze=False)
for col, (nombre, exp) in enumerate(ALG_LIST):
    fed_auc_per_h = np.mean(
        [[s['final_clients'][k]['auc'] for k in range(4)] for s in exp['seeds']], axis=0
    )
    delta_fed_local = [fed_auc_per_h[k] - local_auc[k+1] for k in range(4)]
    delta_cent_fed  = [cent_auc[k+1] - fed_auc_per_h[k] for k in range(4)]

    axes[0][col].bar(range(4), delta_fed_local, color='tab:green')
    axes[0][col].axhline(0, color='black', lw=1)
    axes[0][col].set_xticks(range(4)); axes[0][col].set_xticklabels(HOSPITALS, rotation=20)
    axes[0][col].set_ylabel('d_fed-local (AUC)')
    axes[0][col].set_title(f'{nombre}: AUC fed - AUC local')
    axes[0][col].grid(axis='y', alpha=0.3)

    axes[1][col].bar(range(4), delta_cent_fed, color='tab:red')
    axes[1][col].axhline(0, color='black', lw=1)
    axes[1][col].set_xticks(range(4)); axes[1][col].set_xticklabels(HOSPITALS, rotation=20)
    axes[1][col].set_ylabel('d_cent-fed (AUC)')
    axes[1][col].set_title(f'{nombre}: AUC cent - AUC fed')
    axes[1][col].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(RESULTS / 'brechas.png', dpi=150)
plt.show()

# Bloque 5. Dinamica del entrenamiento

## 12. Curvas de AUC global por ronda

In [ ]:
n_alg = len(ALG_LIST)
fig, axes = plt.subplots(1, n_alg, figsize=(5 * n_alg, 4.5), squeeze=False)
axes = axes[0]

for ax, (nombre, exp) in zip(axes, ALG_LIST):
    for i, seed_res in enumerate(exp['seeds']):
        h = seed_res['history']
        ax.plot(h['round'], h['global_auc'], alpha=0.5, label=f'seed {i+1}')
    mean_auc = np.mean([s['history']['global_auc'] for s in exp['seeds']], axis=0)
    ax.plot(exp['seeds'][0]['history']['round'], mean_auc, 'k--', lw=2, label='media')
    ax.set_title(f'{nombre} - AUC global por ronda')
    ax.set_xlabel('Ronda'); ax.set_ylabel('AUC-ROC')
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(RESULTS / 'curvas_auc.png', dpi=150)
plt.show()

## 13. Curvas de perdida BCE por ronda

`train_loss` es la perdida local promedio reportada por los clientes; `global_val_loss` es la perdida BCE del modelo agregado sobre el test global. La perdida tiene resolucion infinita aunque AUC sature.

In [ ]:
n_alg = len(ALG_LIST)
fig, axes = plt.subplots(1, n_alg, figsize=(5 * n_alg, 4.5), squeeze=False)
axes = axes[0]

for ax, (nombre, exp) in zip(axes, ALG_LIST):
    rounds      = exp['seeds'][0]['history']['round']
    train_loss  = np.mean([s['history']['train_loss']      for s in exp['seeds']], axis=0)
    val_loss    = np.mean([s['history']['global_val_loss'] for s in exp['seeds']], axis=0)
    ax.plot(rounds[1:], train_loss, 'o-', label='train (media seeds)', alpha=0.85)
    ax.plot(rounds,     val_loss,   's-', label='val global (media seeds)', alpha=0.85)
    ax.set_title(f'{nombre} - perdida BCE por ronda')
    ax.set_xlabel('Ronda'); ax.set_ylabel('Perdida BCE')
    ax.legend(); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(RESULTS / 'curvas_perdida.png', dpi=150)
plt.show()

## 14. AUC del modelo agregado por hospital y por ronda

Las secciones anteriores muestran el modelo agregado en el test global mezclado, y sigma(AUC) entre hospitales como resumen de la dispersion. Aqui se ven las **trayectorias individuales** de cada hospital: una linea por hospital, una figura por algoritmo. Permite responder preguntas como "FedProx mejora a H3 (sagital) durante el entrenamiento, o solo cierra la brecha al final?".

Tambien se añade el centralizado para tener la cota superior por hospital.


In [ ]:
fig, axes = plt.subplots(1, len(ALG_LIST) + 1, figsize=(5 * (len(ALG_LIST) + 1), 4.5), squeeze=False)
axes = axes[0]

# Centralizado: 1 sola seed pero traza por hospital y por ronda
ax = axes[0]
if 'per_client_auc' in central['history']:
    rounds_c = central['history']['round']
    pcauc = np.array(central['history']['per_client_auc'])  # shape [R, 4]
    for k in range(4):
        ax.plot(rounds_c, pcauc[:, k], marker='o', label=HOSPITALS[k])
    ax.set_title('Centralizado (cota superior)')
    ax.set_xlabel('Ronda'); ax.set_ylabel('AUC del modelo en cada hospital')
    ax.legend(fontsize=8); ax.grid(alpha=0.3); ax.set_ylim(0.4, 1.02)
else:
    ax.text(0.5, 0.5, 'centralized.json sin per_client_auc.\nReentrena.', ha='center', va='center')
    ax.set_title('Centralizado')

# Federados
for i, (nombre, exp) in enumerate(ALG_LIST, start=1):
    ax = axes[i]
    seeds = exp['seeds']
    if 'per_client_auc' not in seeds[0]['history']:
        ax.text(0.5, 0.5, f'{nombre}.json sin per_client_auc.\nReentrena.', ha='center', va='center')
        ax.set_title(nombre)
        continue
    rounds = seeds[0]['history']['round']
    pcauc = np.array([s['history']['per_client_auc'] for s in seeds])  # [n_seeds, R, 4]
    pcauc_mean = pcauc.mean(axis=0)
    pcauc_std  = pcauc.std(axis=0)
    for k in range(4):
        ax.plot(rounds, pcauc_mean[:, k], marker='o', label=HOSPITALS[k])
        ax.fill_between(rounds, pcauc_mean[:, k] - pcauc_std[:, k], pcauc_mean[:, k] + pcauc_std[:, k], alpha=0.15)
    ax.set_title(f'{nombre} (modelo global por hospital)')
    ax.set_xlabel('Ronda'); ax.set_ylabel('AUC')
    ax.legend(fontsize=8); ax.grid(alpha=0.3); ax.set_ylim(0.4, 1.02)

plt.tight_layout()
plt.savefig(RESULTS / 'auc_por_hospital_y_ronda.png', dpi=150)
plt.show()

# Tabla complementaria: AUC por hospital en la primera y la ultima ronda
filas = []
for nombre, exp in ALG_LIST:
    seeds = exp['seeds']
    if 'per_client_auc' not in seeds[0]['history']:
        continue
    pcauc = np.array([s['history']['per_client_auc'] for s in seeds])
    pcauc_mean = pcauc.mean(axis=0)
    rounds = seeds[0]['history']['round']
    for k in range(4):
        filas.append({
            'Algoritmo': nombre, 'Hospital': HOSPITALS[k],
            'AUC ronda 1': f"{pcauc_mean[1, k]:.4f}" if len(rounds) > 1 else '-',
            'AUC ronda final': f"{pcauc_mean[-1, k]:.4f}",
            'delta entrenamiento': f"{pcauc_mean[-1, k] - pcauc_mean[1, k]:+.4f}" if len(rounds) > 1 else '-',
        })
if filas:
    display(pd.DataFrame(filas))


## 15. Convergencia temprana (rondas 0-5)

Aunque ambos algoritmos terminen con AUC alto, la velocidad de convergencia distingue las familias del T8. Bajo cambio de dominio visual entre planos, FedAvg sufre client drift en las primeras rondas; el termino proximal de FedProx (T8.2.3) y el momento de FedAvgM (T8.3.3) deberian suavizar esa fase.

In [ ]:
N_EARLY = 6
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

panels = [
    ('global_auc', 'AUC global', axes[0]),
    ('global_val_loss', 'Perdida BCE val global', axes[1]),
    ('equity_auc', 'sigma(AUC) entre hospitales', axes[2]),
]

linestyles = ['-', '--', ':']
for key, ylabel, ax in panels:
    for (nombre, exp), ls in zip(ALG_LIST, linestyles):
        rounds_full = exp['seeds'][0]['history']['round']
        vals_mean = np.mean([s['history'][key] for s in exp['seeds']], axis=0)
        vals_std  = np.std([s['history'][key] for s in exp['seeds']], axis=0)
        rounds = rounds_full[:N_EARLY]
        m = vals_mean[:N_EARLY]
        s = vals_std[:N_EARLY]
        ax.plot(rounds, m, ls=ls, lw=2, label=nombre)
        ax.fill_between(rounds, m - s, m + s, alpha=0.2)
    ax.set_xlabel('Ronda'); ax.set_ylabel(ylabel)
    ax.set_title(f'{ylabel} - rondas tempranas')
    ax.legend(); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(RESULTS / 'convergencia_temprana.png', dpi=150)
plt.show()

print('Ronda en la que se alcanza por primera vez un AUC objetivo (media seeds):')
print(f"{'algoritmo':12s} {'>=0.90':>8s} {'>=0.95':>8s} {'>=0.99':>8s}")
for nombre, exp in ALG_LIST:
    auc_mean = np.mean([s['history']['global_auc'] for s in exp['seeds']], axis=0)
    rounds = exp['seeds'][0]['history']['round']
    fila = [nombre]
    for thr in (0.90, 0.95, 0.99):
        idx = next((i for i, a in enumerate(auc_mean) if a >= thr), None)
        fila.append(f"{rounds[idx]}" if idx is not None else "-")
    print(f"{fila[0]:12s} {fila[1]:>8s} {fila[2]:>8s} {fila[3]:>8s}")

## 16. Tiempo por ronda (T9.4.10)

Tiempo wall-clock acumulado al final de cada ronda. T9.3.10 advierte que el wall-clock de simulacion **no equivale** al de un sistema federado real (no hay latencia de red ni stragglers); aqui se usa solo como proxy comparativo.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
linestyles = ['-', '--', ':']
for (nombre, exp), ls in zip(ALG_LIST, linestyles):
    if 'wall_time' not in exp['seeds'][0]['history']:
        continue
    rounds   = exp['seeds'][0]['history']['round']
    wt_seeds = np.array([s['history']['wall_time'] for s in exp['seeds']])
    wt_mean  = wt_seeds.mean(axis=0)
    wt_std   = wt_seeds.std(axis=0)
    ax.plot(rounds, wt_mean, ls=ls, lw=2, label=nombre)
    ax.fill_between(rounds, wt_mean - wt_std, wt_mean + wt_std, alpha=0.2)

ax.set_title('Tiempo wall-clock acumulado (s) por ronda')
ax.set_xlabel('Ronda'); ax.set_ylabel('Tiempo acumulado (s)')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(RESULTS / 'tiempo_por_ronda.png', dpi=150)
plt.show()

filas_t = []
for nombre, exp in ALG_LIST:
    if 'wall_time' not in exp['seeds'][0]['history']:
        continue
    totals = [s['history']['wall_time'][-1] for s in exp['seeds']]
    filas_t.append({
        'Algoritmo': nombre,
        'Tiempo total (s) media': f"{np.mean(totals):.1f}",
        'Tiempo total (s) std':   f"{np.std(totals):.1f}",
        'Tiempo medio/ronda (s)': f"{np.mean(totals) / len(exp['seeds'][0]['history']['round']):.1f}",
    })
tabla_t = pd.DataFrame(filas_t)
display(tabla_t)

# Bloque 6. Analisis fino

## 17. Diferencia pareada FedX - FedAvg

Cada seed se entrena con FedAvg, FedProx y FedAvgM con la misma inicializacion. Eso permite el analisis pareado: para cada hospital y cada seed se calcula la diferencia respecto a FedAvg como baseline. Positivo significa mejora frente al baseline.

In [ ]:
seeds_avg = fedavg['seeds']
other_algs = [(n, e) for n, e in ALG_LIST if n != 'FedAvg']

filas_delta = []
for nombre_other, exp_other in other_algs:
    seeds_other = exp_other['seeds']
    n_seeds = min(len(seeds_avg), len(seeds_other))
    for k in range(4):
        fila = {'Comparacion': f'{nombre_other} - FedAvg', 'Hospital': HOSPITALS[k]}
        for metric_key, label in [('auc', 'dAUC'), ('f1', 'dF1'), ('sensitivity', 'dSen'), ('specificity', 'dSpe'), ('accuracy', 'dAcc')]:
            diffs = [seeds_other[i]['final_clients'][k].get(metric_key, float('nan'))
                     - seeds_avg[i]['final_clients'][k].get(metric_key, float('nan'))
                     for i in range(n_seeds)]
            diffs = [d for d in diffs if not np.isnan(d)]
            fila[label] = '-' if not diffs else (f"{np.mean(diffs):+.4f}" + (f" ± {np.std(diffs):.4f}" if len(diffs) > 1 else ""))
        filas_delta.append(fila)
    fila_g = {'Comparacion': f'{nombre_other} - FedAvg', 'Hospital': 'Global'}
    for metric_key, label in [('auc', 'dAUC'), ('f1', 'dF1'), ('sensitivity', 'dSen'), ('specificity', 'dSpe'), ('accuracy', 'dAcc')]:
        diffs = [seeds_other[i]['final_global'].get(metric_key, float('nan'))
                 - seeds_avg[i]['final_global'].get(metric_key, float('nan'))
                 for i in range(n_seeds)]
        diffs = [d for d in diffs if not np.isnan(d)]
        fila_g[label] = '-' if not diffs else (f"{np.mean(diffs):+.4f}" + (f" ± {np.std(diffs):.4f}" if len(diffs) > 1 else ""))
    filas_delta.append(fila_g)

tabla_delta = pd.DataFrame(filas_delta)
print('Diferencias respecto a FedAvg (positivo = mejora frente al baseline):')
display(tabla_delta)

fig, axes = plt.subplots(1, max(len(other_algs), 1), figsize=(7 * max(len(other_algs), 1), 4), squeeze=False)
axes = axes[0]
x = np.arange(4); width = 0.27
for ax, (nombre_other, exp_other) in zip(axes, other_algs):
    seeds_other = exp_other['seeds']
    n_seeds = min(len(seeds_avg), len(seeds_other))
    for i, (key, label) in enumerate([('auc', 'dAUC'), ('f1', 'dF1'), ('sensitivity', 'dSen')]):
        means, stds = [], []
        for k in range(4):
            diffs = [seeds_other[s]['final_clients'][k][key] - seeds_avg[s]['final_clients'][k][key]
                     for s in range(n_seeds)]
            means.append(np.mean(diffs)); stds.append(np.std(diffs))
        ax.bar(x + (i-1)*width, means, width, yerr=stds, capsize=4, label=label)
    ax.axhline(0, color='black', lw=1)
    ax.set_xticks(x); ax.set_xticklabels(HOSPITALS)
    ax.set_ylabel(f'{nombre_other} - FedAvg')
    ax.set_title(f'Diferencia pareada {nombre_other} - FedAvg por hospital')
    ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(RESULTS / 'delta_vs_fedavg.png', dpi=150)
plt.show()

## 18. Metricas adicionales post-hoc: AUPRC, Brier, ECE, F1 con threshold optimo

`evaluate()` guarda las probabilidades crudas del modelo final. Eso permite calcular sin reentrenar:

- **AUPRC**: area bajo PR curve, mas sensible al desbalance que AUC-ROC.
- **Brier score**: calibracion de las probabilidades respecto a las labels.
- **ECE**: Expected Calibration Error con bins.
- **F1 con threshold optimo**: barrer thresholds y elegir el que maximiza F1 sobre el test global.

In [ ]:
def expected_calibration_error(probs, labels, n_bins=10):
    bin_edges = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    n = len(probs)
    for i in range(n_bins):
        m = (probs >= bin_edges[i]) & (probs < bin_edges[i+1])
        if i == n_bins - 1:
            m = (probs >= bin_edges[i]) & (probs <= bin_edges[i+1])
        if m.sum() == 0:
            continue
        bin_conf = probs[m].mean()
        bin_acc  = labels[m].mean()
        ece += (m.sum() / n) * abs(bin_conf - bin_acc)
    return float(ece)


def best_f1_threshold(probs, labels):
    thresholds = np.linspace(0.05, 0.95, 19)
    best_t, best_f1 = 0.5, -1.0
    for t in thresholds:
        preds = (probs >= t).astype(int)
        f1 = f1_score(labels, preds, zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, t
    return best_t, best_f1


filas_extra = []
for nombre, exp in ALG_LIST:
    auprcs, briers, eces, f1opts, t_opts = [], [], [], [], []
    for s in exp['seeds']:
        probs  = np.array(s['final_global'].get('probs', []))
        labels = np.array(s['final_global'].get('labels', [])).astype(int)
        if len(probs) == 0:
            continue
        auprcs.append(average_precision_score(labels, probs))
        briers.append(brier_score_loss(labels, probs))
        eces.append(expected_calibration_error(probs, labels))
        t, f1 = best_f1_threshold(probs, labels)
        t_opts.append(t); f1opts.append(f1)
    if not auprcs:
        continue
    filas_extra.append({
        'Algoritmo': nombre,
        'AUPRC': f"{np.mean(auprcs):.4f} ± {np.std(auprcs):.4f}",
        'Brier': f"{np.mean(briers):.4f} ± {np.std(briers):.4f}",
        'ECE':   f"{np.mean(eces):.4f} ± {np.std(eces):.4f}",
        'thr* (max F1)': f"{np.mean(t_opts):.2f}",
        'F1 con thr*': f"{np.mean(f1opts):.4f} ± {np.std(f1opts):.4f}",
    })

probs_c  = np.array(central['final_global'].get('probs', []))
labels_c = np.array(central['final_global'].get('labels', [])).astype(int)
if len(probs_c):
    t_c, f1_c = best_f1_threshold(probs_c, labels_c)
    filas_extra.insert(0, {
        'Algoritmo': 'Centralizado',
        'AUPRC': f"{average_precision_score(labels_c, probs_c):.4f}",
        'Brier': f"{brier_score_loss(labels_c, probs_c):.4f}",
        'ECE':   f"{expected_calibration_error(probs_c, labels_c):.4f}",
        'thr* (max F1)': f"{t_c:.2f}",
        'F1 con thr*':   f"{f1_c:.4f}",
    })

tabla_extra = pd.DataFrame(filas_extra)
tabla_extra.to_csv(RESULTS / 'metricas_extra.csv', index=False)
display(tabla_extra)

# Bloque 7. Coste y cierre

## 19. Coste de comunicacion

T9.3.10: en simulacion local conviene reportar **rondas y bytes/ronda** como metricas comparables; el wall-clock solo si se aclara que es de simulacion.

In [ ]:
n_params = fedavg['seeds'][0]['n_params']
comm_mb_per_round_per_client = fedavg['seeds'][0]['comm_bytes_per_round'] / 1e6
N_CLIENTS = 4
ROUNDS = len(fedavg['seeds'][0]['history']['round']) - 1

print(f'Parametros del modelo:           {n_params:,}')
print(f'Bytes por ronda y cliente:       {comm_mb_per_round_per_client:.2f} MB')
print(f'Bytes por ronda total ({N_CLIENTS} clientes): {comm_mb_per_round_per_client * N_CLIENTS:.2f} MB')
print(f'Coste total ({ROUNDS} rondas):           {comm_mb_per_round_per_client * N_CLIENTS * ROUNDS:.2f} MB')
print()
print('Mismo tamano de parametros -> todas las variantes (FedAvg/Prox/AvgM) tienen igual coste de comunicacion.')

## 20. Cierre y limitaciones

**Lo que sostiene el experimento:**

- Tres familias de T8.4.11 cubiertas: FedAvg (base), FedProx (correccion cliente, T8.2), FedAvgM (correccion servidor, T8.3.3).
- Baselines local y centralizado segun T9.3.7.
- Metricas T9.4.7: AUC, F1, Sensibilidad, Especificidad, Accuracy + AUPRC, Brier, ECE post-hoc.
- Equidad T9.4.6: sigma(AUC) por ronda, peor cliente, ratio peor/mejor, brecha mediana-peor.
- Brechas T9.4.8: Delta_fed-local y Delta_cent-fed.
- Heterogeneidad T6.6: histograma marginal + embeddings ResNet-18 (estructural).

**Limitaciones que conviene reportar honestamente:**

1. **Tarea trivialmente separable.** ResNet-18 preentrenada en ImageNet ya da AUC > 0.7 sin fine-tune. AUC satura cerca de 1 a las pocas rondas; hay que leer F1, Sensibilidad y equidad como discriminadores reales.
2. **KDE oculta heterogeneidad de pixel.** El preprocesado del proveedor normaliza la intensidad por imagen; la heterogeneidad visible para un humano no aparece en el histograma marginal (TV ~ 0.02). En cambio en embeddings ResNet-18 si emerge (Frechet ~ 50-150).
3. **Solo 2 seeds.** Suficiente para reportar dispersion pero insuficiente para t-test riguroso.
4. **Asimetria train/test.** Train con ratio MS ~0.605 frente a test ~0.459. Threshold 0.5 esta descalibrado; por eso F1 con threshold optimo aporta una vista mas justa.
5. **Sin clientes no vistos** (T9.4.3). Con 4 hospitales no es viable held-out de cliente entero; no se puede reclamar generalizacion a hospitales nuevos.
6. **Wall-clock simulado.** No equivale al despliegue real (sin latencia de red, sin stragglers).